# Bước 3 — Suy luận và độ trễ trên Lightning AI
Chọn Studio với **1 GPU T4**, mở JupyterLab. Upload notebook này và **step3_bundle.zip**
vào cùng thư mục rồi chạy các ô từ trên xuống. Không cần PyTorch Lightning Trainer.
Notebook chỉ khoảng 15 KB; ZIP chứa duy nhất checkpoint T05, khoảng 117 MB.

Không train lại, không đánh giá test. Giữ nguyên 9 cấu hình: 1-view, hflip prob/logit,
5-crop, 256/288/320, temperature, AMP. Mỗi cấu hình đo batch 1/32,
10 warmup + 100 lần đồng bộ GPU. Input sẵn ở GPU; gồm TTA/model/softmax/T.

Dùng bộ torch/torchvision CUDA đang hoạt động trong Studio và timm==1.0.30.
Môi trường suy luận ghi riêng vào runtime.json, không sửa cấu hình train T05.
Đối chiếu **toàn bộ 3.501 logit val** với cache T05 trước khi tái sử dụng.
Chỉ gộp số đo trong cùng môi trường Lightning, không lấy số Colab lấp vào.

Dữ liệu/kết quả được lưu trong deepweeds_step3_lightning dưới thư mục làm việc.
Chạy lại ô sẽ dùng lại kết quả đã hoàn thành. Cuối notebook tải step3_artifacts.zip
qua liên kết hoặc Download trong Files của JupyterLab. Lưu thêm notebook có output.


In [ ]:
import sys, subprocess, json, base64, hashlib, shutil, os
from pathlib import Path
if sys.version_info < (3,10):
    raise RuntimeError('Notebook yêu cầu Python >=3.10.')
# Keep the CUDA-enabled torch/torchvision supplied by the Studio.
try:
    import torch, torchvision
except ImportError as exc:
    raise RuntimeError('Thiếu torch/torchvision. Trong terminal Studio cài: '
        'pip install torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu126 '
        'rồi restart kernel và chạy lại.') from exc
if not torch.cuda.is_available():
    raise RuntimeError('Bật machine T4 và chọn kernel có CUDA rồi chạy lại. '
        'Nếu đang dùng torch CPU, cài bộ CUDA trong hướng dẫn và restart kernel.')
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__,
      '| torchvision:', torchvision.__version__, '| CUDA:', torch.version.cuda)
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'timm==1.0.30', 'fvcore==0.1.5.post20221221', 'matplotlib', 'pandas',
    'pillow', 'scikit-learn', 'openpyxl'], capture_output=True, text=True)
print(result.stdout, result.stderr)
result.check_returncode()
if 'timm' in sys.modules and sys.modules['timm'].__version__ != '1.0.30':
    raise RuntimeError('Đã cài timm 1.0.30; restart kernel rồi chạy lại từ đầu.')
NOTEBOOK_DIR = Path.cwd().resolve()
BASE_DIR = NOTEBOOK_DIR / 'deepweeds_step3_lightning'
PROJECT_DIR = BASE_DIR / 'project'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print('Thư mục lưu dữ liệu/kết quả:', BASE_DIR)


In [ ]:
import zipfile
def stream_sha256(stream):
    digest = hashlib.sha256()
    for chunk in iter(lambda: stream.read(1 << 20), b''):
        digest.update(chunk)
    return digest.hexdigest()
EXPECTED_BUNDLE_SHA256 = '10a543adbcbc04a4724c974b2375d6db9734fa0c1af9aba8bf22999db7ab4376'
BUNDLE_PATH = Path(os.environ.get('DEEPWEEDS_BUNDLE', str(NOTEBOOK_DIR / 'step3_bundle.zip'))).expanduser().resolve()
if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(f'Upload step3_bundle.zip vào thư mục {NOTEBOOK_DIR}, hoặc đặt biến DEEPWEEDS_BUNDLE bằng đường dẫn ZIP.')
with BUNDLE_PATH.open('rb') as stream:
    if stream_sha256(stream) != EXPECTED_BUNDLE_SHA256:
        raise ValueError('ZIP không khớp notebook; thay bằng ZIP đi kèm.')
with zipfile.ZipFile(BUNDLE_PATH) as bundle:
    for info in bundle.infolist():
        if info.is_dir():
            continue
        target = (PROJECT_DIR / info.filename).resolve()
        if not target.is_relative_to(PROJECT_DIR.resolve()):
            raise ValueError('Đường dẫn ZIP vượt ngoài dự án.')
        if info.filename == 'results.xlsx' and target.exists():
            continue
        if target.exists() and info.filename == 'step3/temperature.json':
            current = json.loads(target.read_text(encoding='utf-8'))
            supplied = json.loads(bundle.read(info))
            if any(current[key] != supplied[key] for key in ('T','fit_split','n','test_used')):
                raise FileExistsError(f'Temperature đã lưu khác ZIP: {target}')
            # Diagnostic floats can vary slightly across NumPy versions; keep T.
            continue
        if target.exists() and info.filename.endswith('.md') and info.filename.startswith(('step1/','step2/')):
            # Local reports may have been enriched after the original export.
            # They are analysis documents; measured source files below are locked.
            target.write_bytes(bundle.read(info))
            continue
        if target.exists() and info.filename.startswith(('runs/','curves/','predictions/','step1/','step2/','step3/source.json')):
            with target.open('rb') as current, bundle.open(info) as supplied:
                if stream_sha256(current) != stream_sha256(supplied):
                    raise FileExistsError(f'Bằng chứng cũ khác ZIP: {target}')
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with bundle.open(info) as supplied, target.open('wb') as destination:
            shutil.copyfileobj(supplied, destination)
assert hashlib.sha256((PROJECT_DIR / 'eval.py').read_bytes()).hexdigest() == '1210f3c7ca1cb9b4488383f1ceeaa139bfdf5c683694b300f6c55b6b9cdde28e'
CODE_DIR = PROJECT_DIR / 'code'
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
for name in ['dataset','losses','model','train','step0','step1','step2','inference','benchmark','step3']:
    sys.modules.pop(name, None)
import dataset, model, train, step0, step1, step2, inference, benchmark, step3
reference = step3.reference(PROJECT_DIR)
runtime = step3.assert_environment(PROJECT_DIR, platform='Lightning AI')
print('Môi trường suy luận:', runtime)
print('Nguồn train T05 giữ nguyên; cần đối chiếu logits val trước khi suy luận.')


In [ ]:
command = [sys.executable, '-X', 'utf8', '-c',
    "import torch,unittest; torch.set_num_threads(2); "
    "s=unittest.defaultTestLoader.discover('tests'); "
    "r=unittest.TextTestRunner(verbosity=1).run(s); raise SystemExit(not r.wasSuccessful())"]
result = subprocess.run(command, cwd=PROJECT_DIR, capture_output=True, text=True)
print(result.stdout, result.stderr)
result.check_returncode()


In [ ]:
from IPython.display import display, Image as DisplayImage
rows = step3.prepare_offline(PROJECT_DIR)
step3.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(step1.read_json(PROJECT_DIR / 'step3/temperature.json'), ensure_ascii=False, indent=2))
display(DisplayImage(filename=str(PROJECT_DIR / 'step3/calibration_reliability.png')))
print('Đã chuẩn bị I00/I07 từ cache; latency vẫn để trống đến khi đo T4.')


In [ ]:
IMAGES_DIR, LABELS_DIR = step0.prepare_data(BASE_DIR)
network = step3.load_selected_model(PROJECT_DIR, device='cuda')
error = step3.verify_model(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, network)
print('Reload T05: sai số logit toàn bộ val:', error)
print(step1.read_json(PROJECT_DIR / 'step3/model_check.json'))
print('ConvNeXt không có BN: fusion không áp dụng. Chỉ tạo val loader.')


## I00 — Mốc FP32 1-view
Đo batch 1 và 32, 100 lần sau 10 warmup.


In [ ]:
row = step3.run_method(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'I00', network)
step3.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(row, ensure_ascii=False, indent=2))


## I01 — TTA lật ngang, gộp xác suất
Đo batch 1 và 32, 100 lần sau 10 warmup.


In [ ]:
row = step3.run_method(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'I01', network)
step3.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(row, ensure_ascii=False, indent=2))


## I02 — 5 crop, gộp xác suất
Đo batch 1 và 32, 100 lần sau 10 warmup.


In [ ]:
row = step3.run_method(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'I02', network)
step3.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(row, ensure_ascii=False, indent=2))


## I03 — Cùng hai view I01, gộp logit
Đo batch 1 và 32, 100 lần sau 10 warmup.


In [ ]:
row = step3.run_method(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'I03', network)
step3.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(row, ensure_ascii=False, indent=2))


## I04 — Input 256
Đo batch 1 và 32, 100 lần sau 10 warmup.


In [ ]:
row = step3.run_method(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'I04', network)
step3.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(row, ensure_ascii=False, indent=2))


## I09 — Input 288
Đo batch 1 và 32, 100 lần sau 10 warmup.


In [ ]:
row = step3.run_method(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'I09', network)
step3.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(row, ensure_ascii=False, indent=2))


## I10 — Input 320
Đo batch 1 và 32, 100 lần sau 10 warmup.


In [ ]:
row = step3.run_method(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'I10', network)
step3.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(row, ensure_ascii=False, indent=2))


## I07 — Temperature scaling 1-view
Đo batch 1 và 32, 100 lần sau 10 warmup.


In [ ]:
row = step3.run_method(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'I07', network)
step3.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(row, ensure_ascii=False, indent=2))


## I08 — AMP FP16 1-view
Đo batch 1 và 32, 100 lần sau 10 warmup.


In [ ]:
row = step3.run_method(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'I08', network)
step3.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(row, ensure_ascii=False, indent=2))


In [ ]:
rows = step3.collect_results(PROJECT_DIR)
display(__import__('pandas').DataFrame(rows))
selection = step3.write_report(PROJECT_DIR)
step3.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(selection, ensure_ascii=False, indent=2))
display(DisplayImage(filename=str(PROJECT_DIR / 'step3/accuracy_latency.png')))
print((PROJECT_DIR / 'step3/step3_report.md').read_text(encoding='utf-8'))


In [ ]:
archive = step3.export_artifacts(PROJECT_DIR, BASE_DIR / 'step3_artifacts.zip')
print('Đã tạo:', archive)
print('ZIP có Excel, report/scatter/reliability, T, runtime, logits/probs/dự đoán val và raw timing.')
from IPython.display import FileLink
display(FileLink(str(archive.relative_to(NOTEBOOK_DIR))))
print('Nếu liên kết không tải được: trong Files mở deepweeds_step3_lightning, '
      'chuột phải step3_artifacts.zip → Download.')


Lưu notebook có output bằng Save, rồi Download từ Files của JupyterLab.
Tải **deepweeds_step3_lightning/step3_artifacts.zip** và gửi về thư mục dự án.
Không có checkpoint mới, không cần tải lại T05.

Bước 3 hoàn thành khi đủ 9 cấu hình, 18 số đo batch, ECE trước/sau T và scatter,
có ít nhất một cấu hình p95 ≤100 ms. Ngưỡng này chỉ áp dụng pipeline GPU đã khai báo.
Mọi quyết định trên val; test giữ đến Bước 4. Không cần chạy lại Bước 1/2 vì đổi nền tảng.
